# Do AI models remember Indian company financials?
### Pilot study, notebook 1 of 2: design and data collection

**How to use this notebook:** run one cell at a time (Shift+Enter). Read the text above each cell *before* you run it. Every design choice is explained, because a professor will ask you about each one.

**Today you stop at the CHECKPOINT cell.** Don't run the full collection until we've checked the smoke-test output together.

## Part A: the study in one page

**Question.** If you ask an AI model for an Indian listed company's revenue and net profit for a past year, does it give the real number from memory? Does that depend on how big the company is?

**Why it matters.** Researchers and funds now use AI models to "predict" stock returns or earnings and then backtest those predictions on past data. If the model already *memorized* what happened, the backtest is cheating without anyone noticing. That is **look-ahead bias**. Lopez-Lira, Tang and Zhu (2025, *The Memorization Problem*) showed GPT-4o recalls US macro data and index levels almost perfectly before its training cutoff, and that memorization is stronger for prominent stocks. Our angle: **Indian companies, across size tiers from ₹8.9 lakh crore (TCS) down to ₹123 crore (Owais Metal).**

**Sample.** 79 NSE/BSE companies with March year-ends and four years of data (FY23 to FY26), from the sector model. Four were dropped: LTIMindtree (latest year is FY25 on Screener), R Systems and ABB India (December year-ends), and Allsec (stale data). FY23 is missing for two companies: Linde India (its FY23 covers 15 months) and Owais Metal (not yet listed, figures under ₹1 crore).

**Size tiers (market cap, Oct 2026).** Large ≥ ₹1,00,000 cr (17 companies) · Mid ₹30,000–1,00,000 cr (20) · Small ₹5,000–30,000 cr (28) · Micro < ₹5,000 cr (14). The cut-offs roughly track how AMFI separates large, mid and small caps.

**The design trick: two models with different training cutoffs.** Indian companies publish annual results in April–May after the March year-end.

| Model (free API) | Training cutoff | FY23 (results Apr–May 2023) | FY24 (Apr–May 2024) | FY25 (Apr–May 2025) | FY26 (Apr–May 2026) |
|---|---|---|---|---|---|
| `openai/gpt-oss-120b` on Groq | 1 Jun 2024 | before | at the edge | **after**: cannot know | after |
| `gemini-3.6-flash` on Google | Mar 2026 (Google says some areas only to Jan 2025) | before | before | before | **after**: cannot know |

Read it like a staircase. **FY23** is a year both models could have seen, so it shows each model's best case. **FY25** is a year one model *could* have seen and the other *could not*. If Gemini is accurate on FY25 and gpt-oss isn't, while both are fine on FY23 and both fail on FY26, the gap is memory, not a better or worse model. The years a model cannot know act as the **control**.

*Why FY23 was added (21 Oct):* the first smoke test showed gpt-oss refusing even FY24. Without FY23, gpt-oss had no year clearly before its cutoff, so "gpt-oss is simply a weaker model" couldn't be ruled out.

**Two prompt styles.**
- **Strict:** "answer from memory; return null if you don't know." Measures *whether the model knows what it knows*.
- **Best estimate:** "always give a number, plus a 0–100 confidence." Gives an answer for every cell, so we can compare accuracy across tiers without models skipping hard ones.

**Main metric (next notebook):** share of answers within 5% of the true figure, by tier × year × model. Revenue is the headline number. Net profit is secondary because its definition varies (attributable vs total, exceptional items) and small profits make percentage errors explode.

**Settings that keep it honest:** temperature 0 (most deterministic), low reasoning effort (we want recall, not a long chain of estimation), no web search tools, and every raw response saved with a timestamp so anyone can re-check it.

## Part B: setup

**One-time, before running anything:**
1. Google AI Studio → *Get API key* (free; no card needed). Groq console → *API Keys* (free).
2. In Colab: click the 🔑 icon in the left sidebar → *Add new secret* → name `GEMINI_API_KEY`, paste the key, toggle *Notebook access* on. Do the same for `GROQ_API_KEY`.
   *Never paste keys into a cell or push them to GitHub. Secrets keep them out of the notebook file.*
3. In Google Drive create the folder `MyDrive/llm_memorization_pilot/` and upload `ground_truth.csv` into it.

The cell below connects Colab to your Drive so results survive if Colab disconnects (Colab wipes its own disk when the session ends).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, time, re, datetime
import requests
import pandas as pd

FOLDER = '/content/drive/MyDrive/llm_memorization_pilot'
RAW_LOG = f'{FOLDER}/raw_responses.jsonl'   # every API response gets appended here
GEMINI_KEY = userdata.get('GEMINI_API_KEY')
GROQ_KEY = userdata.get('GROQ_API_KEY')
print('keys loaded:', bool(GEMINI_KEY), bool(GROQ_KEY))

### Load the ground truth
Figures are ₹ crore, consolidated, from Screener.in (pulled 19 Oct 2026). `rev_` = revenue from operations ("Sales" on Screener), `np_` = net profit (Screener includes minority interest).

The `audit` column marks 12 companies you will check by hand tomorrow: two **suspects** (Vedanta's FY25 looks restated after its demerger; Coal India's FY25 "Sales" jump looks off) and a **random** stratified sample of 10. Screener shows the latest restated numbers. A model learned the *first-reported* numbers. That gap is a real research issue called **point-in-time data**, and a professor will love that you caught it.

In [ ]:
truth = pd.read_csv(f'{FOLDER}/ground_truth.csv')
print(len(truth), 'companies')
print(truth['tier'].value_counts().sort_index())
truth[['company', 'exchange_label', 'tier', 'mcap_cr', 'rev_fy23', 'rev_fy24', 'rev_fy25', 'rev_fy26', 'audit']].head(10)

## Part C: the prompts

Read these carefully; the wording *is* the method.
- We name the company **and** its exchange code so there is no confusion (e.g. Cyient vs Cyient DLM).
- We spell out each fiscal year's end date, because "FY24" can mean different things in different countries.
- We force units (**₹ crore**). A common failure is answering in ₹ million or US$. We will detect those in scoring rather than hide them.
- We ask for JSON so a script can read the answer with no human judgement involved.

In [ ]:
SYSTEM = {
    'strict': (
        "You are answering from memory only. You have no internet access and must not look anything up. "
        "If you do not remember a figure, return null for it. Do not estimate or extrapolate."
    ),
    'best': (
        "You are answering from memory only. You have no internet access and must not look anything up. "
        "Give your best numeric answer for every figure even if unsure, and a confidence from 0 to 100 "
        "for each fiscal year that the figures are correct to within 5%."
    ),
}

def user_prompt(row, variant):
    conf = ', "confidence": 0-100' if variant == 'best' else ''
    nulls = ' (use null if unknown)' if variant == 'strict' else ''
    return (
        f"Company: {row['company']} (listed in India; {row['exchange_label']}).\n"
        "Give this company's CONSOLIDATED revenue from operations and CONSOLIDATED net profit, "
        f"in Indian rupees crore (INR crore){nulls}, for these fiscal years:\n"
        "- FY23 = fiscal year ended 31 March 2023\n"
        "- FY24 = fiscal year ended 31 March 2024\n"
        "- FY25 = fiscal year ended 31 March 2025\n"
        "- FY26 = fiscal year ended 31 March 2026\n"
        "Reply with JSON only, exactly in this shape:\n"
        '{"FY23": {"revenue": number, "net_profit": number' + conf + '}, '
        '"FY24": {"revenue": number, "net_profit": number' + conf + '}, '
        '"FY25": {"revenue": number, "net_profit": number' + conf + '}, '
        '"FY26": {"revenue": number, "net_profit": number' + conf + '}}'
    )

print(SYSTEM['strict'], '\n')
print(user_prompt(truth.iloc[0], 'strict'))

## Part D: talking to the two APIs

Each function sends one HTTP request and returns the raw text. Note what we **don't** do: we pass no search or "grounding" tools, so the model can only use what is in its weights.

`call_with_retry` handles the free tier's bad moods. "Too many requests" (HTTP 429), "overloaded" (HTTP 503) and timeouts all get a retry after a growing pause. A call that still fails is logged as an error and simply retried the next time you run the collection cell.

**Why the Gemini settings changed (21 Oct):** in the first smoke test Gemini was overloaded (503) and slow (over 2 minutes per answer). Gemini 3 models "think" before answering, and thinking takes time. We now ask for the *minimal* thinking level first (falling back to *low*, then the default if the model refuses a level) and wait up to 3 minutes per call. Minimal thinking also fits the study: we want recall, not a long chain of estimation. The level actually used is saved with every answer.

In [ ]:
MODELS = {
    'gpt-oss-120b':     {'provider': 'groq',   'id': 'openai/gpt-oss-120b', 'pause': 2.5, 'timeout': 120, 'tries': 5},
    'gemini-3.6-flash': {'provider': 'gemini', 'id': 'gemini-3.6-flash',    'pause': 7.0, 'timeout': 180, 'tries': 4},
}
GEMINI_LEVELS = ['minimal', 'low', None]     # thinking levels to try, in order
_gemini_level_ok = {}                        # remembers which level each Gemini model accepted

def call_groq(model_id, system, user, timeout, json_mode=True):
    body = {'model': model_id, 'temperature': 0, 'reasoning_effort': 'low',
            'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]}
    if json_mode:
        body['response_format'] = {'type': 'json_object'}
    r = requests.post('https://api.groq.com/openai/v1/chat/completions',
                      headers={'Authorization': f'Bearer {GROQ_KEY}'}, json=body, timeout=timeout)
    if r.status_code == 400 and json_mode:          # some models reject JSON mode: retry without it
        return call_groq(model_id, system, user, timeout, json_mode=False)
    r.raise_for_status()
    d = r.json()
    return d['choices'][0]['message']['content'], d.get('model', model_id), {'reasoning': 'low'}

def call_gemini(model_id, system, user, timeout):
    levels = [_gemini_level_ok[model_id]] if model_id in _gemini_level_ok else GEMINI_LEVELS
    for lvl in levels:
        cfg = {'temperature': 0, 'responseMimeType': 'application/json'}
        if lvl:
            cfg['thinkingConfig'] = {'thinkingLevel': lvl}
        body = {'systemInstruction': {'parts': [{'text': system}]},
                'contents': [{'role': 'user', 'parts': [{'text': user}]}],
                'generationConfig': cfg}
        r = requests.post(f'https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent',
                          headers={'x-goog-api-key': GEMINI_KEY}, json=body, timeout=timeout)
        if r.status_code == 400 and lvl is not None and model_id not in _gemini_level_ok:
            print(f'  {model_id} refused thinking level {lvl!r}: {r.text[:120]}')
            continue                                   # try the next level
        r.raise_for_status()
        _gemini_level_ok[model_id] = lvl
        d = r.json()
        parts = d['candidates'][0]['content']['parts']
        text = ''.join(p.get('text', '') for p in parts if not p.get('thought'))
        thoughts = d.get('usageMetadata', {}).get('thoughtsTokenCount', 0)
        return text, d.get('modelVersion', model_id), {'thinking': lvl, 'thought_tokens': thoughts}
    r.raise_for_status()

def call_with_retry(model_name, system, user):
    m = MODELS[model_name]
    fn = call_groq if m['provider'] == 'groq' else call_gemini
    wait = 20
    for attempt in range(m['tries']):
        try:
            return fn(m['id'], system, user, m['timeout'])
        except requests.HTTPError as e:
            why, retry = f'HTTP {e.response.status_code}', e.response.status_code in (429, 500, 502, 503, 504)
        except (requests.Timeout, requests.ConnectionError):
            why, retry = 'timeout', True
        if not retry or attempt == m['tries'] - 1:
            raise RuntimeError(f'{model_name}: gave up after {attempt + 1} tries ({why})')
        print(f'  {model_name}: {why}, waiting {wait}s'); time.sleep(wait); wait *= 2

def parse_json(text):
    """Model output -> dict. Falls back to the first {...} block if there is extra text around it."""
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text or '', re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    return None

### Quick health check of Gemini (run this before the smoke test)

One tiny request per Gemini model and thinking level, with no retries, to see which combination is up and fast right now. You want **200** and a few seconds. A 503 means Google's free servers are overloaded at the moment; 400 means that model refuses that thinking level.

In [ ]:
for mid in ['gemini-3.8-flash', 'gemini-3.6-flash']:
    for lvl in ['minimal', 'low']:
        body = {'contents': [{'role': 'user', 'parts': [{'text': 'Reply with this JSON only: {"ok": true}'}]}],
                'generationConfig': {'temperature': 0, 'thinkingConfig': {'thinkingLevel': lvl}}}
        t0 = time.time()
        try:
            r = requests.post(f'https://generativelanguage.googleapis.com/v1beta/models/{mid}:generateContent',
                              headers={'x-goog-api-key': GEMINI_KEY}, json=body, timeout=90)
            note = r.text[:100].replace(chr(10), ' ') if r.status_code != 200 else \
                   f"thought tokens: {r.json().get('usageMetadata', {}).get('thoughtsTokenCount', 0)}"
            print(f'{mid:18s} {lvl:8s} HTTP {r.status_code}  {time.time() - t0:5.1f}s  {note}')
        except requests.Timeout:
            print(f'{mid:18s} {lvl:8s} timed out after 90s')

## Part E: smoke test (3 companies × 2 models, strict prompt)

Before spending ~300 API calls, we test on one company from each end of the size range plus one in between. **Look at the raw text, not just the parsed numbers:** is it valid JSON? Right units? Does it return null for FY26?

In [ ]:
def run_one(model_name, variant, row):
    t0 = time.time()
    rec = {'model': model_name, 'variant': variant, 'code': row['code'], 'company': row['company'],
           'ts': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}
    try:
        text, version, extra = call_with_retry(model_name, SYSTEM[variant], user_prompt(row, variant))
        rec.update(status='ok', raw=text, model_version=version, **extra)
    except Exception as e:
        rec.update(status='error', raw=str(e)[:500], model_version=None)
    rec['seconds'] = round(time.time() - t0, 1)
    return rec

smoke = truth[truth['code'].isin(['TCS', 'KAYNES', 'MINDTECK'])]
for _, row in smoke.iterrows():
    print('=' * 80)
    print(row['company'], '| TRUE revenue FY23/24/25/26:', row['rev_fy23'], row['rev_fy24'], row['rev_fy25'], row['rev_fy26'],
          '| TRUE net profit:', row['np_fy23'], row['np_fy24'], row['np_fy25'], row['np_fy26'])
    for model_name in MODELS:
        rec = run_one(model_name, 'strict', row)
        print(f'\n[{model_name} | {rec["status"]} | {rec["seconds"]}s | version={rec["model_version"]} | thinking={rec.get("thinking", "-")}]')
        print(rec['raw'][:600])
        time.sleep(MODELS[model_name]['pause'])

## ✋ CHECKPOINT: stop here today

Send me two things:

**1. The full output of the smoke-test cell** (copy-paste it).

**2. Your answers to these five questions, in your own words, 2–3 lines each.** These are the questions a professor will ask in the first five minutes. Don't look anything up; answer from what you read above. If you can't answer one, say so; that is the point.

1. What is look-ahead bias, and how would an AI model's memory cause it in a backtest?
2. Why use two models with *different* training cutoffs instead of the single best model?
3. Why is revenue the headline number and net profit only secondary?
4. What does the strict prompt (null allowed) measure that the best-estimate prompt can't, and the other way round?
5. Give two reasons a model's answer could disagree with our ground truth *even if the model remembers correctly*.

## Step 2, as actually run (21–22 Oct 2026): batched collection

**What changed and why.** Google's free tier allows only **20 requests per day** per Gemini model. One company per request would need 158 Gemini requests, so we ask about **10 companies per request** (8 requests per model and prompt, 48 in total). All three models get exactly the same batched questions, so the comparison stays fair. We first planned Gemini 3.8 Flash, but its 20 requests were used up by the early tests and it was badly overloaded (503 errors, 80+ seconds per answer); **Gemini 3.6 Flash** has the same March 2026 training cutoff and answered in about 2 seconds.

The cell is resume-safe: a re-run only asks about companies still missing. Each re-run of a missing batch costs one of Gemini's 20 daily requests.

**Two practical additions.** Models did not always return the requested shape (Flash-Lite sometimes returned a list of company blocks), so `parse_json` and `to_map` normalise every shape we saw. Gemini 3.6 Flash finished on 22 October after its daily quota reset; Gemini 3.5 Flash-Lite (same cutoff, smaller) was added as a second Gemini model.

In [ ]:
BATCH = 10
BATCH_LOG = f'{FOLDER}/raw_batched.jsonl'
MODELS = {
    'gpt-oss-120b':          {'provider': 'groq',   'id': 'openai/gpt-oss-120b',   'pause': 3,  'timeout': 180, 'tries': 5},
    'gemini-3.6-flash':      {'provider': 'gemini', 'id': 'gemini-3.6-flash',      'pause': 15, 'timeout': 240, 'tries': 4},
    'gemini-3.5-flash-lite': {'provider': 'gemini', 'id': 'gemini-3.5-flash-lite', 'pause': 5,  'timeout': 180, 'tries': 4},
}

def parse_json(text):
    """Model output -> dict or list. Tries the whole text, then every [...] or {...} block, then per-company blocks."""
    text = text or ''
    for cand in [text] + re.findall(r'\[.*\]|\{.*\}', text, re.S):
        try:
            return json.loads(cand)
        except Exception:
            pass
    out = {}
    pattern = r'"([A-Za-z0-9&\-]+)"\s*:\s*(\{\s*"FY23".*?"FY26"\s*:\s*\{[^{}]*\}\s*\})'
    for code, body in re.findall(pattern, text, re.S):
        try:
            out[code] = json.loads(body)
        except Exception:
            pass
    return out or None

def to_map(d):
    """Normalise the shapes models actually returned into {company code: {FY23: ..., ...}}."""
    codes = set(truth['code'].astype(str))
    if isinstance(d, dict):
        k, v = next(iter(d.items())) if len(d) == 1 else (None, None)
        if k is not None and k not in codes and isinstance(v, (dict, list)):
            return to_map(v)          # e.g. {"companies": {...}}
        return d
    out = {}
    for item in d if isinstance(d, list) else []:
        if not isinstance(item, dict):
            continue
        code = item.get('code') or item.get('company_code') or item.get('ticker')
        if code is not None:
            out[str(code)] = {k: v for k, v in item.items() if str(k).upper().startswith('FY')}
        else:
            for k, v in item.items():
                if isinstance(v, dict):
                    out[str(k)] = v
    return out

def batch_prompt(rows, variant):
    conf = ', "confidence": 0-100' if variant == 'best' else ''
    nulls = ' Use null for any figure you do not remember.' if variant == 'strict' else ''
    yr = '{"revenue": number, "net_profit": number' + conf + '}'
    shape = '{"FY23": ' + yr + ', "FY24": ' + yr + ', "FY25": ' + yr + ', "FY26": ' + yr + '}'
    lines = '\n'.join(f"- {r['code']}: {r['company']} ({r['exchange_label']})" for _, r in rows.iterrows())
    return ("For each Indian listed company below, give its CONSOLIDATED revenue from operations and "
            "CONSOLIDATED net profit, in Indian rupees crore (INR crore), for FY23 (fiscal year ended 31 March 2023), "
            "FY24 (ended 31 March 2024), FY25 (ended 31 March 2025) and FY26 (ended 31 March 2026)." + nulls +
            "\n" + lines + "\nReply with JSON only: one key per company code (exactly as written above), "
            "each value in exactly this shape: " + shape)

done = set()
if os.path.exists(BATCH_LOG):
    for line in open(BATCH_LOG):
        r = json.loads(line)
        if r['status'] == 'ok':
            done.add((r['model'], r['variant'], r['code']))
truth['code'] = truth['code'].astype(str)
chunks = [truth.iloc[i:i + BATCH] for i in range(0, len(truth), BATCH)]

for m in MODELS:
    for v in ('strict', 'best'):
        for full in chunks:
            ch = full[[(m, v, c) not in done for c in full['code']]]   # only companies still missing
            if ch.empty:
                continue
            t0 = time.time()
            try:
                text, version, extra = call_with_retry(m, SYSTEM[v], batch_prompt(ch, v))
                d, status = to_map(parse_json(text)), 'ok'
            except Exception as e:
                text, version, extra, d, status = str(e), None, {}, {}, 'error'
            n_ok = 0
            with open(BATCH_LOG, 'a') as f:
                for _, r in ch.iterrows():
                    ans = d.get(r['code'])
                    good = status == 'ok' and isinstance(ans, dict)
                    n_ok += good
                    rec = {'model': m, 'variant': v, 'code': r['code'], 'company': r['company'],
                           'ts': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                           'status': 'ok' if good else 'error',
                           'raw': json.dumps(ans) if good else str(text)[:300],
                           'model_version': version, 'batch_size': len(ch), **extra}
                    f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            print(f'{m:17s} {v:6s} {ch["code"].iloc[0]:>10s} … {ch["code"].iloc[-1]:<10s} {n_ok}/{len(ch)} ok  {time.time() - t0:.0f}s')
            time.sleep(MODELS[m]['pause'])

log = pd.DataFrame([json.loads(l) for l in open(BATCH_LOG)])
ok = log[log['status'] == 'ok'].drop_duplicates(['model', 'variant', 'code'])
print('\nAnswers saved (should be 79 each):')
print(ok.groupby(['model', 'variant']).size())

## Step 2 (original plan, NOT used): one company per request

79 companies × 2 models × 2 prompt styles = **316 calls**. Groq's half takes about 10 minutes. Gemini's half depends on how busy Google's free servers are: anywhere from 30 minutes to a couple of hours. Errors don't stop the run; they're logged and filled in when you run the cell again. The loop is **resume-safe**: every response is appended to `raw_responses.jsonl` in your Drive, and finished (model, prompt, company) combinations are skipped. If Colab disconnects, just run the cell again.

In [ ]:
def done_keys():
    keys = set()
    if os.path.exists(RAW_LOG):
        for line in open(RAW_LOG):
            r = json.loads(line)
            if r['status'] == 'ok':
                keys.add((r['model'], r['variant'], r['code']))
    return keys

finished = done_keys()
todo = [(m, v, row) for m in MODELS for v in ('strict', 'best') for _, row in truth.iterrows()
        if (m, v, row['code']) not in finished]
print(len(finished), 'already done;', len(todo), 'to go')

with open(RAW_LOG, 'a') as f:
    for i, (m, v, row) in enumerate(todo, 1):
        rec = run_one(m, v, row)
        f.write(json.dumps(rec, ensure_ascii=False) + '\n'); f.flush()
        if i % 20 == 0 or rec['status'] != 'ok':
            print(f'{i}/{len(todo)}  {m:17s} {v:6s} {row["code"]:12s} {rec["status"]}')
        time.sleep(MODELS[m]['pause'])
print('finished')

### Sanity check before scoring
Every (model, prompt) pair should have 79 successful responses and very few unparseable ones. If not, re-run the cell above: it only fills the gaps.

In [ ]:
log = pd.DataFrame([json.loads(l) for l in open(RAW_LOG)])
log = log.sort_values('ts').drop_duplicates(['model', 'variant', 'code', 'status'], keep='last')
ok = log[log['status'] == 'ok'].drop_duplicates(['model', 'variant', 'code'], keep='last').copy()
ok['parsed'] = ok['raw'].apply(parse_json).notna()
print(ok.groupby(['model', 'variant']).agg(responses=('code', 'count'), parsed=('parsed', 'sum')))
print('\nmodel versions actually served:', ok['model_version'].unique())

**Next (notebook 2):** turning raw answers into results. We'll score errors, catch unit mistakes (₹ million vs ₹ crore), make the tier × year × model table and the two charts that go in the write-up.